# CherenkovLens Walkthrough Demo
### End-to-End IACT Simulation, Image Cleaning, Hillas Moment Extraction & Gamma/Hadron Discrimination

This notebook demonstrates the complete CherenkovLens analysis pipeline modelled on the analysis chain of the **MACE telescope** (Hanle, Ladakh) and the **HAGAR array**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cherenkovlens.camera import CameraGeometry
from cherenkovlens.simulate import ShowerSimulator
from cherenkovlens.cleaning import tailcuts_clean, naive_clean
from cherenkovlens.hillas import calculate_hillas_parameters
from cherenkovlens.utils import load_config, set_seed

# Set master seed for reproducibility
rng = set_seed(42)
cfg = load_config()
camera = CameraGeometry(**cfg.get("camera", {}))
simulator = ShowerSimulator(camera=camera, config=cfg, rng=rng)

print(f"Camera initialized: {camera.n_pixels} pixels ({camera.n_pixels_side}x{camera.n_pixels_side}), pixel size = {camera.pixel_size_deg} deg")

## 1. Shower Simulation
We simulate both an electromagnetic shower (gamma-ray) and a hadronic shower (proton) with Night Sky Background (NSB) photo-electron noise.

In [ ]:
# Simulate on-axis gamma event
gamma_ev = simulator.simulate_gamma(amplitude=850.0, x0=0.45, y0=0.45, length=0.25, width=0.08, sigma_nsb=4.5)

# Simulate multi-blob proton event
proton_ev = simulator.simulate_proton(amplitude=950.0, sigma_nsb=4.5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = axes[0].imshow(gamma_ev.image, origin="lower", cmap="magma")
axes[0].set_title("Gamma-ray Shower (Signal)")
plt.colorbar(im0, ax=axes[0], label="pe")

im1 = axes[1].imshow(proton_ev.image, origin="lower", cmap="magma")
axes[1].set_title("Proton Shower (Hadron Background)")
plt.colorbar(im1, ax=axes[1], label="pe")
plt.tight_layout()
plt.show()

## 2. Image Cleaning: Two-Level Tail-Cuts vs Naive Threshold
We compare the dual-threshold island connected-component algorithm against single hard thresholding.

In [ ]:
tc_res = tailcuts_clean(gamma_ev.image, sigma_nsb=4.5, core_sigma=6.0, boundary_sigma=3.0, min_island_size=2)
naive_res = naive_clean(gamma_ev.image, threshold=4.5 * 4.5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(gamma_ev.image, origin="lower", cmap="magma")
axes[0].set_title("Raw Noisy Image")

axes[1].imshow(naive_res.cleaned_image, origin="lower", cmap="magma")
axes[1].set_title("Naive Single Threshold")

axes[2].imshow(tc_res.cleaned_image, origin="lower", cmap="magma")
axes[2].set_title("Two-Level Tail-Cuts (6/3 sigma)")
plt.tight_layout()
plt.show()

## 3. Hillas Parameter & Timing Extraction
We reconstruct the geometrical image moments and arrival-time features from the cleaned shower image.

In [ ]:
hillas = calculate_hillas_parameters(tc_res.cleaned_image, camera=camera, time_image=gamma_ev.time_image)

print("=== Extracted Hillas & Timing Parameters ===")
for k, v in hillas.to_feature_dict().items():
    print(f"{k:15s}: {v:.4f}")

## 4. Machine Learning Classification & Q-Factor Optimization
We evaluate the trained Random Forest classifier on surviving events and observe the Quality Factor curve.

In [ ]:
from cherenkovlens.classify import train_and_evaluate_classifier
from cherenkovlens.dataset import generate_dataset

# Generate a quick evaluation set
df_quick = generate_dataset(n_events=1200, gamma_fraction=0.5, seed=42, show_progress=False, n_jobs=1)
pipeline, results = train_and_evaluate_classifier(df_quick, model_type="random_forest", random_state=42)

print(f"Test ROC AUC: {results.auc:.4f}")
print(f"Peak Quality Factor Q: {results.best_q_factor:.2f} at cut = {results.best_threshold:.2f}")
print(f"Gamma efficiency: {results.gamma_efficiency_at_best_q * 100:.1f}%, Hadron survival: {results.proton_survival_at_best_q * 100:.2f}%")